# 01. 탐색적 데이터 분석 (EDA)
> 난임 환자 임신 성공 여부 예측

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글 폰트
# Mac 사용자는 아래 줄로 교체
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = '../data/'
train = pd.read_csv(DATA_PATH + 'train.csv')
test  = pd.read_csv(DATA_PATH + 'test.csv')
print('train:', train.shape)
print('test: ', test.shape)

## 1. 기본 정보

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

## 2. 타겟 분포 확인 (클래스 불균형 체크)

In [ ]:
# 타겟 컬럼명 확인 후 수정 필요
target_col = '임신 성공 여부'  # 실제 컬럼명으로 수정

print(train[target_col].value_counts())
print('\n성공 비율:', train[target_col].mean())

fig, ax = plt.subplots(figsize=(6, 4))
train[target_col].value_counts().plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'])
ax.set_title('타겟 분포 (임신 성공 여부)')
ax.set_xlabel('성공 여부')
ax.set_ylabel('count')
plt.tight_layout()
plt.show()

## 3. 결측치 분석

In [ ]:
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
missing_df = pd.DataFrame({'결측 수': missing, '결측률(%)': missing_pct})
missing_df = missing_df[missing_df['결측 수'] > 0].sort_values('결측률(%)', ascending=False)
print(missing_df)

In [ ]:
# 결측률 시각화
fig, ax = plt.subplots(figsize=(10, 8))
missing_df['결측률(%)'].plot(kind='barh', ax=ax, color='#3498db')
ax.set_title('컬럼별 결측률')
ax.set_xlabel('결측률 (%)')
plt.tight_layout()
plt.show()

## 4. 나이 분포 vs 임신 성공률

In [ ]:
age_success = train.groupby('시술 당시 나이')[target_col].agg(['mean', 'count'])
age_success.columns = ['성공률', '건수']
print(age_success.sort_values('성공률', ascending=False))

fig, ax = plt.subplots(figsize=(10, 5))
age_order = ['만18-34세', '만35-37세', '만38-39세', '만40-42세', '만43-44세', '만45세 이상']
age_order_exist = [a for a in age_order if a in age_success.index]
age_success.loc[age_order_exist, '성공률'].plot(kind='bar', ax=ax, color='#9b59b6')
ax.set_title('나이대별 임신 성공률')
ax.set_ylabel('성공률')
ax.set_xlabel('나이')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. 시술 유형별 성공률

In [ ]:
print(train.groupby('시술 유형')[target_col].agg(['mean', 'count']))
print('\n')
print(train.groupby('특정 시술 유형')[target_col].agg(['mean', 'count']))

## 6. 수치형 피처 상관관계

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
if target_col in num_cols:
    corr = train[num_cols].corr()[target_col].drop(target_col).sort_values(ascending=False)
    print('타겟과 상관관계 TOP 20')
    print(corr.head(20))
    print('\n타겟과 상관관계 BOTTOM 10')
    print(corr.tail(10))